In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
%matplotlib inline
print("Libraries imported!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
%matplotlib inline
print("Libraries imported!")

In [ ]:
# 1. Create 'Car_Age' feature
CURRENT_YEAR = 2024
df['Car_Age'] = CURRENT_YEAR - df['Year']

# 2. Extract 'Brand' from Car_Name (e.g., 'Maruti Swift Dzire' -> 'Maruti')
df['Brand'] = df['Car_Name'].str.split().str[0]

# Drop the original 'Year' and 'Car_Name' as we have extracted their value
df = df.drop(['Year', 'Car_Name'], axis=1)

display(df.head())

Feature Engineering:
I created Car_Age by subtracting the manufacturing year from the current year, as depreciation is heavily tied to age rather than the raw year. I also extracted the Brand from the car name, as brand reputation significantly impacts resale value.

In [ ]:
# 1. Distribution of Selling Price
plt.figure(figsize=(8,5))
sns.histplot(df['Selling_Price'], kde=True, bins=30, color='blue')
plt.title('Distribution of Car Selling Prices')
plt.xlabel('Selling Price (Lakhs)')
plt.show()

# 2. Price vs Fuel Type
plt.figure(figsize=(8,5))
sns.boxplot(x='Fuel_Type', y='Selling_Price', data=df, palette='Set2')
plt.title('Selling Price vs Fuel Type')
plt.show()

# 3. Price vs Car Age
plt.figure(figsize=(8,5))
sns.scatterplot(x='Car_Age', y='Selling_Price', data=df, alpha=0.6, color='red')
plt.title('Selling Price vs Car Age')
plt.xlabel('Car Age (Years)')
plt.show()

# 4. Correlation Heatmap (Numeric only)
plt.figure(figsize=(8,6))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Define Features (X) and Target (y)
X = df.drop('Selling_Price', axis=1)
y = df['Selling_Price']

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify categorical and numerical columns
cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(exclude='object').columns

# Create a preprocessor: Scale numbers, One-Hot Encode categories
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

In [ ]:
# Function to evaluate models
def evaluate_model(name, model):
    # Create a pipeline (preprocess -> model)
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    
    # Train
    pipe.fit(X_train, y_train)
    
    # Predict
    y_pred = pipe.predict(X_test)
    
    # Metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print(f"--- {name} ---")
    print(f"MAE:  {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R2:   {r2:.3f}\n")
    
    return pipe

# 1. Linear Regression
lr_pipe = evaluate_model("Linear Regression", LinearRegression())

# 2. Random Forest Regressor
rf_pipe = evaluate_model("Random Forest", RandomForestRegressor(n_estimators=100, random_state=42))

In [ ]:
# Assuming Random Forest performed better (it usually does on this data)
best_model = rf_pipe.named_steps['regressor']
encoder = rf_pipe.named_steps['preprocessor'].named_transformers_['cat']

# Get feature names after One-Hot Encoding
cat_features = encoder.get_feature_names_out(cat_cols)
all_features = list(num_cols) + list(cat_features)

# Get importances
importances = best_model.feature_importances_

# Plot top 10 features
feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='viridis')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.xlabel('Importance Score')
plt.show()

Model Comparison & Conclusion:
The Random Forest Regressor outperformed Linear Regression, achieving a higher R² score and lower RMSE. This indicates that the relationship between car features and price is non-linear.

Feature Impact: According to the feature importance chart, Present_Price (the original showroom price) and Car_Age are the strongest predictors of the current selling price.